# Autograds

Autograd is PyTorch's automatic differentiation engine that helps compute gradients for tensor operations, which is essential for training neural networks. Here's a quick overview:

1. **Automatic Differentiation**: It automatically calculates gradients (derivatives) of tensor operations, which is crucial for backpropagation in deep learning.

2. **How it works**:
- Tracks all operations on tensors with requires_grad=True
- Builds a computation graph (directed acyclic graph) of operations
- Uses the chain rule to compute gradients during the backward pass

In [6]:
import torch
# The autograd package provides automatic differentiation 
# for all operations on Tensors

# requires_grad = True -> tracks all operations on the tensor. 
x = torch.randn(3, requires_grad=True)
y = x + 2
print(x) # created by the user -> grad_fn is None
print(y)
print(y.grad_fn)

tensor([-0.4144,  1.6877,  2.0356], requires_grad=True)
tensor([1.5856, 3.6877, 4.0356], grad_fn=<AddBackward0>)


In [3]:
z = y * y * 3
print(z)
z = z.mean()
print(z)

tensor([20.7868,  3.2374, 35.0889], grad_fn=<MulBackward0>)
tensor(19.7044, grad_fn=<MeanBackward0>)


In [ ]:
z.backward()
print(x.grad) # dz/dx Gradient of z with respect to x

tensor([5.2646, 2.0776, 6.8400])


Non scalar values

In [ ]:
# Model with non-scalar output:
# If a Tensor is non-scalar (more than 1 elements), we need to specify arguments for backward() 
# specify a gradient argument that is a tensor of matching shape.
# needed for vector-Jacobian product

x = torch.randn(3, requires_grad=True)

y = x * 2
for _ in range(10):
    y = y * 2

print(y)
print(y.shape)

v = torch.tensor([0.1, 1.0, 0.0001], dtype=torch.float32)
y.backward(v)
print(x.grad)


tensor([ 1691.0112, -2981.7588, -2312.4067], grad_fn=<MulBackward0>)
torch.Size([3])
tensor([2.0480e+02, 2.0480e+03, 2.0480e-01])


# Stop a tensor from tracking history:

In [10]:
# For example during our training loop when we want to update our weights
# then this update operation should not be part of the gradient computation
# - x.requires_grad_(False)
# - x.detach()
# - wrap in 'with torch.no_grad():'

# .requires_grad_(...) changes an existing flag in-place.
a = torch.randn(2, 2)
print(a.requires_grad)
b = ((a * 3) / (a - 1))
print(b.grad_fn)
a.requires_grad_(True)
print(a.requires_grad)
b = (a * a).sum()
print(b.grad_fn)

# .detach(): get a new Tensor with the same content but no gradient computation:
a = torch.randn(2, 2, requires_grad=True)
print(a.requires_grad)
b = a.detach()
print(b.requires_grad)

# wrap in 'with torch.no_grad():'
a = torch.randn(2, 2, requires_grad=True)
print(a.requires_grad)
with torch.no_grad():
    print((x ** 2).requires_grad)

False
None
True
True
False
True
False


# with and wtihout grad_zero

In [12]:
weights = torch.ones(4, requires_grad=True)

for epoch in range(3):
    # just a dummy example
    model_output = (weights*3).sum()
    model_output.backward()
    
    print(weights.grad)

    # optimize model, i.e. adjust weights...
    with torch.no_grad():
        weights -= 0.1 * weights.grad

    # this is important! It affects the final weights & output
    # weights.grad.zero_()

tensor([3., 3., 3., 3.])
tensor([6., 6., 6., 6.])
tensor([9., 9., 9., 9.])


In [13]:
weights = torch.ones(4, requires_grad=True)

for epoch in range(3):
    # just a dummy example
    model_output = (weights*3).sum()
    model_output.backward()
    
    print(weights.grad)

    # optimize model, i.e. adjust weights...
    with torch.no_grad():
        weights -= 0.1 * weights.grad

    # this is important! It affects the final weights & output
    weights.grad.zero_()

tensor([3., 3., 3., 3.])
tensor([3., 3., 3., 3.])
tensor([3., 3., 3., 3.])
